# Threat Intelligence & Scam Detection Exploration
This notebook demonstrates the end-to-end Machine Learning pipeline for text-based threat classification.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pickle
import os

### Step 1: Load and Inspect Dataset

In [2]:
# Load dataset from data folder
data_path = '../data/data.csv' if os.path.exists('../data/data.csv') else 'data.csv'
df = pd.read_csv(data_path)
print(f"Total records: {len(df)}")
df.head()

### Step 2: Feature Extraction (TF-IDF)

In [3]:
X = df['message']
y = df['label']

vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
X_vec = vectorizer.fit_transform(X)
print("TF-IDF Shape:", X_vec.shape)

### Step 3: Train Binary Model (SAFE vs SCAM)

In [4]:
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_vec, y)
print(f"Binary Label Accuracy: {model.score(X_vec, y):.2%}")

### Step 4: Train Category Model (SCAM Subtypes)

In [5]:
scam_df = df[df['label'] == 'SCAM'].dropna(subset=['category'])
X_scam = scam_df['message']
y_scam = scam_df['category']

X_scam_vec = vectorizer.transform(X_scam)
cat_model = LogisticRegression(max_iter=300, random_state=42)
cat_model.fit(X_scam_vec, y_scam)
print(f"Category Model Accuracy: {cat_model.score(X_scam_vec, y_scam):.2%}")

### Step 5: Test Inference on New Payload

In [6]:
test_message = "Congratulations you won a lottery of 50000 click link to claim reward now!"
test_vec = vectorizer.transform([test_message])
pred_label = model.predict(test_vec)[0]
pred_cat = cat_model.predict(test_vec)[0] if pred_label == 'SCAM' else 'N/A'
confidence = np.max(model.predict_proba(test_vec))

print(f"Payload: {test_message}")
print(f"Predicted Label: {pred_label} ({confidence:.1%})")
print(f"Predicted Category: {pred_cat}")